# 01 · Data audit & exploratory analysis (T04)

AISC5030 Topic 3 — daily air pollution patterns at the Aotizhongxin station.

This notebook **calls the research modules in `src/air_quality_clustering`** and reads authoritative run outputs; it does not re-implement cleaning or plotting logic. Execute top-to-bottom after `--stage prepare eda` has produced a run.

Protocol references: `docs/05_data_dictionary.md`, `docs/06_experiment_protocol.md`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

def latest_run(stage):
    runs = []
    for d in (ROOT / 'results' / 'runs').iterdir():
        mp = d / 'manifest.json'
        if not mp.is_file():
            continue
        m = json.loads(mp.read_text(encoding='utf-8'))
        done = m.get('stages_completed') or m.get('stage_summary', {}).get('stages_completed', [])
        if stage in done and m.get('status') in ('completed', 'completed_with_warnings'):
            runs.append((d.name, d, m))
    if not runs:
        raise RuntimeError(f'no completed run contains stage {stage}; run scripts/run_project.py first')
    return max(runs, key=lambda t: t[0])

from air_quality_clustering.config import load_config
cfg = load_config(ROOT / 'configs' / 'project.yaml')
run_name, RUN, manifest = latest_run('eda')
print('run_id:', run_name)
print('data version (CSV sha256):', manifest['input_sha256'].get('data/raw/PRSA_Data_Aotizhongxin_20130301-20170228.csv'))
print('stages completed:', manifest.get('stages_completed') or manifest.get('stage_summary', {}).get('stages_completed', []))

## 1. Source and cleaning funnel

Raw file is read-only; the audit below comes from the actual prepare run.

In [ ]:
summary = json.loads((RUN / 'tables' / 'prepare_summary.json').read_text(encoding='utf-8'))
audit = summary['hourly_audit']
print('hourly rows read/written:', audit['rows_read'], audit['rows_written'])
print('missing hours per pollutant:', audit['missing_counts'])
print('negative concentrations (invalidated):', audit['negative_counts'])
print('acceptance checks:', summary['acceptance_checks'])
print('eligible days by year:', summary['eligible_days_by_year'])

In [ ]:
coverage = pd.read_csv(ROOT / 'data' / 'processed' / 'daily_coverage.csv')
daily = pd.read_csv(ROOT / 'data' / 'processed' / 'daily_pollutants.csv')
excluded = pd.read_csv(ROOT / 'data' / 'processed' / 'excluded_days.csv')
print('calendar days:', len(coverage), '| eligible:', int(coverage.eligible.sum()), '| excluded:', len(excluded))
excluded.head(10)

## 2. F01 — monthly coverage

Question: is missingness concentrated in time? The dashed line is the 18/24-hour quality rule (a project rule, not a regulatory standard).

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F01_monthly_coverage.png')))

## 3. F02 — raw vs log1p distributions

Question: why transform? Daily concentrations are right-skewed and span orders of magnitude (CO included, in µg/m³); log1p compresses the tail before standardization.

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F02_distributions.png')))
daily[['PM2.5','PM10','SO2','NO2','CO','O3']].describe(percentiles=[.05,.25,.5,.75,.95]).T.round(2)

## 4. F03 — Spearman correlation

Question: do correlated pollutants double-count information in Euclidean distance? PM2.5/PM10 and NO2/CO correlations are discussed in the report as a limitation of the distance, not corrected silently.

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F03_correlation.png')))
daily[['PM2.5','PM10','SO2','NO2','CO','O3']].corr(method='spearman').round(3)

## 5. F04 — daily time series with gaps

Excluded days remain gaps; no interpolation connects them.

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F04_time_series.png')))

## 6. Observations to test with clustering (not conclusions)

- Do high-particulate days and high-O3 days form different combinations?
- Are winter/summer days separable by concentrations alone (season is NOT an input)?
- Is the structure cluster-like or a continuous gradient? DBSCAN will probe this.

No cluster names or k are fixed here; selection follows the frozen protocol in T08.